# Reasoning-as-Graphs: GNN-Based Detection and Explanation of LLM Reasoning Failures
### Machine Learning with Graphs | Spring 2026 | Rice University
**Authors:** Latthika Selvamurugan, Abhirami Kathirvel  
**Supervisor:** Prof. Arlei Lopes da Silva

---

### Project Overview
We convert LLM Chain-of-Thought reasoning traces into graphs and train a GNN to classify whether the reasoning is correct or wrong. GNNExplainer then identifies exactly which step caused the failure — without any human annotation.

**Pipeline:**
```
Stage 1 (data generation) → Stage 2 (parse steps) → Stage 3 (embed steps)
→ Stage 4 (build graphs) → Stage 5 (train GNN) → Stage 6 (evaluate) → Stage 7 (explain)
```

**Note:** Stage 1 (data generation via Groq API) is excluded from this notebook. Run `Stage1_Batch2.ipynb` separately to regenerate the dataset. This notebook assumes `data/gsm8k_with_traces.csv` already exists.

---

### Setup — run this cell first, then Kernel → Restart


In [ ]:
import sys

# Install all required libraries
# Run once, then Kernel → Restart before proceeding
!{sys.executable} -m pip install -q pandas numpy scikit-learn --user
!{sys.executable} -m pip install -q torch==2.5.1 --index-url https://download.pytorch.org/whl/cpu
!{sys.executable} -m pip install -q sentence-transformers torch-geometric matplotlib seaborn
print("All libraries installed — do Kernel → Restart, then run from Stage 2")


---
## Stage 2: Parse Reasoning Traces into Individual Steps

**Input:** `data/gsm8k_with_traces.csv` — one row per trace  
**Output:** `data/gsm8k_steps.csv` — one row per step  

Each trace is a block of text like:
```
Step 1: Natalia sold 48 clips in April.
Step 2: In May she sold half as many: 48/2 = 24.
Step 3: Total = 48 + 24 = 72.
#### 72
```
We split this into individual rows — one per step — removing the `Step N:` prefix and the `#### answer` line. These individual steps become nodes in Stage 4's graphs.


In [ ]:
import os
import re
import pandas as pd
import numpy as np

DATA_DIR    = os.path.join(os.getcwd(), 'data')
INPUT_FILE  = os.path.join(DATA_DIR, 'gsm8k_with_traces.csv')
OUTPUT_FILE = os.path.join(DATA_DIR, 'gsm8k_steps.csv')

df = pd.read_csv(INPUT_FILE)
print(f"Loaded {len(df)} traces | Correct: {(df['label']==1).sum()} | Wrong: {(df['label']==0).sum()}")


In [ ]:
def parse_trace(trace_text):
    """
    Split a trace string into a list of individual step strings.
    
    Example input:
        'Step 1: She has 48 clips.\nStep 2: Half of 48 = 24.\nStep 3: Total = 72.\n#### 72'
    
    Example output:
        ['Step 1: She has 48 clips.', 'Step 2: Half of 48 = 24.', 'Step 3: Total = 72.']
    """
    if not trace_text or pd.isna(trace_text):
        return []
    
    # Split on 'Step N:' pattern — works for Step 1:, Step 2:, Step 10:, etc.
    # re.split keeps the delimiter when we use a capturing group ()
    parts = re.split(r'(Step\s+\d+:\s*)', str(trace_text))
    
    steps = []
    i = 0
    while i < len(parts):
        # re.split gives us alternating [text, delimiter, text, delimiter, ...]
        # We want to combine each delimiter with the text that follows it
        if re.match(r'Step\s+\d+:\s*', parts[i]):
            # This part is the 'Step N:' header
            step_header = parts[i]
            step_body   = parts[i+1] if i+1 < len(parts) else ''
            full_step   = step_header + step_body
            # Drop the #### answer line if it ended up inside a step body
            full_step = re.sub(r'####.*', '', full_step).strip()
            if full_step:
                steps.append(full_step)
            i += 2
        else:
            # Text before the first Step — usually empty, skip it
            i += 1
    
    # Fallback: if no Step N: pattern found, treat the whole trace as one step
    # (after removing the #### line)
    if len(steps) == 0:
        cleaned = re.sub(r'####.*', '', str(trace_text)).strip()
        if cleaned:
            steps = [cleaned]
    
    return steps


def clean_step(step_text):
    """
    Clean one step string:
    - Remove the 'Step N:' prefix
    - Strip whitespace
    - Return None if empty after cleaning
    
    Example input:  'Step 2: Half of 48 is 24 clips.'
    Example output: 'Half of 48 is 24 clips.'
    """
    if not step_text:
        return None
    
    # Remove 'Step N:' prefix
    cleaned = re.sub(r'^Step\s+\d+:\s*', '', step_text.strip())
    
    # Strip any leftover whitespace
    cleaned = cleaned.strip()
    
    # Return None if nothing left
    return cleaned if cleaned else None


# ── Quick test on a real example ──────────────────────────────────────────
test_trace = """Step 1: Natalia sold 48 clips in April.
Step 2: In May she sold half as many, so 48 / 2 = 24 clips.
Step 3: Total = 48 + 24 = 72 clips.
#### 72"""

print("TEST — parsing a sample trace:")
print("-" * 50)
parsed = parse_trace(test_trace)
print(f"Number of steps found: {len(parsed)}")
print()
for j, step in enumerate(parsed):
    cleaned = clean_step(step)
    print(f"Step {j+1} raw:     {step}")
    print(f"Step {j+1} cleaned: {cleaned}")
    print()

print("✅ Functions working correctly")

In [ ]:
all_steps = []   # Will hold one dict per step
skipped   = 0    # Traces we couldn't parse (missing trace text)

for trace_id, row in df.iterrows():
    
    # Skip rows with no trace text
    if pd.isna(row['trace']) or str(row['trace']).strip() == '':
        skipped += 1
        continue
    
    # Step 1: Parse trace into list of steps
    steps = parse_trace(row['trace'])
    
    # Step 2: Clean each step and save as a row
    valid_steps = []
    for step in steps:
        cleaned = clean_step(step)
        if cleaned:                      # Skip empty steps
            valid_steps.append(cleaned)
    
    # Skip traces that produced zero valid steps
    if len(valid_steps) == 0:
        skipped += 1
        continue
    
    # Step 3: Save each step as one row in all_steps
    for step_num, step_text in enumerate(valid_steps, start=1):
        all_steps.append({
            'trace_id':    trace_id,              # Links back to the parent trace
            'orig_idx':    row['orig_idx'],        # Original GSM8K index
            'split':       row['split'],           # train / val / test
            'question':    row['question'],        # Original question
            'step_num':    step_num,               # Position: 1, 2, 3...
            'step_text':   step_text,              # Cleaned step text
            'total_steps': len(valid_steps),       # Total steps in this trace
            'label':       int(row['label']),      # 1 = correct, 0 = wrong
        })

# Convert to DataFrame
df_steps = pd.DataFrame(all_steps)

print(f"Input traces:    {len(df)}")
print(f"Skipped traces:  {skipped} (missing or unparseable)")
print(f"Output steps:    {len(df_steps)}")
print(f"Avg steps/trace: {len(df_steps) / max(len(df) - skipped, 1):.1f}")
print()
print(f"Label distribution in steps:")
step_counts = df_steps['label'].value_counts()
print(f"  Steps from correct traces (1): {step_counts.get(1, 0)}")
print(f"  Steps from wrong traces   (0): {step_counts.get(0, 0)}")
print()
print("✅ All traces parsed")

In [ ]:
df_steps.to_csv(OUTPUT_FILE, index=False)

print(f"✅ Saved to: {OUTPUT_FILE}")
print(f"   Rows: {len(df_steps)}")
print(f"   Columns: {list(df_steps.columns)}")
print()

# Preview first 5 rows
print("Preview (first 5 rows):")
print("-" * 80)
preview = df_steps[['trace_id', 'split', 'step_num', 'step_text', 'total_steps', 'label']].head(5)
for _, r in preview.iterrows():
    print(f"trace_id={r['trace_id']}  step={r['step_num']}/{r['total_steps']}  label={r['label']}  split={r['split']}")
    print(f"  text: {r['step_text'][:80]}")
    print()

In [ ]:
df_check = pd.read_csv(OUTPUT_FILE)

print("=" * 55)
print("  STAGE 2 QUALITY CHECKS")
print("=" * 55)

all_pass = True

# Check 1: File exists and has rows
print(f"\n[1] Total step rows: {len(df_check)}")
if len(df_check) > 0:
    print("    ✅ PASS")
else:
    print("    ❌ FAIL — no rows")
    all_pass = False

# Check 2: No empty step texts
empty = df_check['step_text'].isna().sum() + (df_check['step_text'].str.strip() == '').sum()
print(f"\n[2] Empty step texts: {empty}")
if empty == 0:
    print("    ✅ PASS")
else:
    print(f"    ⚠️  {empty} empty steps found")
    all_pass = False

# Check 3: Step numbers are sequential per trace
bad_sequences = 0
for tid, group in df_check.groupby('trace_id'):
    expected = list(range(1, len(group) + 1))
    actual   = sorted(group['step_num'].tolist())
    if expected != actual:
        bad_sequences += 1
print(f"\n[3] Traces with non-sequential step numbers: {bad_sequences}")
if bad_sequences == 0:
    print("    ✅ PASS")
else:
    print(f"    ⚠️  {bad_sequences} traces have step numbering issues")
    all_pass = False

# Check 4: All 3 splits present
splits_found = set(df_check['split'].unique())
splits_need  = {'train', 'val', 'test'}
print(f"\n[4] Splits present: {splits_found}")
split_counts = df_check['split'].value_counts()
for s in ['train', 'val', 'test']:
    print(f"    {s}: {split_counts.get(s, 0)} steps")
if splits_need.issubset(splits_found):
    print("    ✅ PASS")
else:
    print(f"    ❌ FAIL — missing splits: {splits_need - splits_found}")
    all_pass = False

# Check 5: Average steps per trace is sensible
avg_steps = df_check.groupby('trace_id')['step_num'].max().mean()
print(f"\n[5] Average steps per trace: {avg_steps:.1f}")
min_steps = df_check.groupby('trace_id')['step_num'].max().min()
max_steps = df_check.groupby('trace_id')['step_num'].max().max()
print(f"    Min: {min_steps}  Max: {max_steps}")
if 2 <= avg_steps <= 10:
    print("    ✅ PASS")
else:
    print("    ⚠️  Unusual step count — check parsing")
    all_pass = False

# Check 6: Both labels present
labels_found = set(df_check['label'].unique())
n_correct = len(df_check[df_check['label']==1])
n_wrong   = len(df_check[df_check['label']==0])
print(f"\n[6] Labels in steps:")
print(f"    Steps from correct traces (1): {n_correct}")
print(f"    Steps from wrong traces   (0): {n_wrong}")
if 0 in labels_found and 1 in labels_found:
    print("    ✅ PASS")
else:
    print("    ❌ FAIL — only one label type found")
    all_pass = False

print()
print("=" * 55)
if all_pass:
    print("  ✅ ALL CHECKS PASSED — Stage 2 complete!")
    print("  ➡️  Ready for Stage 3 (embedding steps)")
else:
    print("  ⚠️  SOME CHECKS FAILED — see above")
print("=" * 55)

---
## Stage 3: Embed Each Step into a 768-Dimensional Vector

**Input:** `data/gsm8k_steps.csv`  
**Output:** `data/gsm8k_embeddings.npy` (shape: n_steps × 768), `data/gsm8k_steps_meta.csv`

We use `all-mpnet-base-v2` from sentence-transformers to convert each step text into a 768-number vector that captures its meaning. Two steps about the same concept will have similar vectors (high cosine similarity). These vectors become the node features in our reasoning graphs.

First run downloads ~420MB — cached after that.


In [ ]:
import numpy as np
import pandas as pd
import os
import time
from sentence_transformers import SentenceTransformer

DATA_DIR   = os.path.join(os.getcwd(), 'data')
INPUT_FILE = os.path.join(DATA_DIR, 'gsm8k_steps.csv')
OUTPUT_NPY = os.path.join(DATA_DIR, 'gsm8k_embeddings.npy')
META_CSV   = os.path.join(DATA_DIR, 'gsm8k_steps_meta.csv')
OUTPUT_CSV = os.path.join(DATA_DIR, 'gsm8k_steps_with_embeddings.csv')

df = pd.read_csv(INPUT_FILE)
print(f"Steps to embed: {len(df)} | Unique traces: {df['trace_id'].nunique()}")


In [ ]:
print("Loading all-mpnet-base-v2...")
print("(First run downloads ~420MB — subsequent runs load from cache instantly)")
print()

model = SentenceTransformer('all-mpnet-base-v2')

print(f"✅ Model loaded")
print()

# Quick test — embed one sentence and check output shape
test_embedding = model.encode("Half of 48 is 24 clips.")
print(f"Test embedding shape: {test_embedding.shape}")
print(f"First 5 values:       {test_embedding[:5].round(4)}")
print()

if test_embedding.shape[0] == 768:
    print("✅ Model outputs 768-dim vectors — correct")
else:
    print(f"⚠️  Unexpected output size: {test_embedding.shape[0]}")

In [ ]:
import time

step_texts = df['step_text'].tolist()

print(f"Embedding {len(step_texts)} steps...")
print(f"Batch size: 64")
print(f"Expected time: ~5–10 minutes on CPU")
print()

start = time.time()

embeddings = model.encode(
    step_texts,
    batch_size=64,
    show_progress_bar=True,    # shows a progress bar
    convert_to_numpy=True      # output as numpy array
)

elapsed = time.time() - start

print()
print(f"✅ Done in {elapsed/60:.1f} minutes")
print(f"   Embeddings shape: {embeddings.shape}")
print(f"   Expected shape:   ({len(step_texts)}, 768)")

if embeddings.shape == (len(step_texts), 768):
    print("   ✅ Shape correct")
else:
    print(f"   ⚠️  Unexpected shape — expected ({len(step_texts)}, 768)")

In [ ]:
# ── Save 1: numpy array (main output for Stage 4+5) ────────────────────────
np.save(OUTPUT_NPY, embeddings)
print(f"✅ Saved embeddings array: {OUTPUT_NPY}")
print(f"   Shape: {embeddings.shape}  |  Size: {os.path.getsize(OUTPUT_NPY)/1e6:.1f} MB")
print()

# ── Save 2: metadata CSV (steps info without embedding columns) ────────────
META_CSV = os.path.join(DATA_DIR, 'gsm8k_steps_meta.csv')
df.to_csv(META_CSV, index=False)
print(f"✅ Saved metadata CSV: {META_CSV}")
print(f"   Rows: {len(df)}  |  Columns: {list(df.columns)}")
print()

# ── Save 3: full CSV with embedding columns (for inspection) ───────────────
emb_cols = pd.DataFrame(
    embeddings,
    columns=[f'emb_{i}' for i in range(embeddings.shape[1])]
)
df_full = pd.concat([df.reset_index(drop=True), emb_cols], axis=1)
df_full.to_csv(OUTPUT_CSV, index=False)
print(f"✅ Saved full CSV: {OUTPUT_CSV}")
print(f"   Rows: {len(df_full)}  |  Total columns: {len(df_full.columns)} (8 meta + 768 emb)")
print()
print("All files saved to data/ folder.")

In [ ]:
from numpy.linalg import norm

# Reload to confirm saved correctly
emb_loaded = np.load(OUTPUT_NPY)
meta       = pd.read_csv(META_CSV)

print("=" * 55)
print("  STAGE 3 QUALITY CHECKS")
print("=" * 55)

all_pass = True

# Check 1: Shape
print(f"\n[1] Embedding shape: {emb_loaded.shape}")
if emb_loaded.shape[0] == len(meta) and emb_loaded.shape[1] == 768:
    print("    ✅ PASS")
else:
    print(f"    ❌ FAIL — expected ({len(meta)}, 768)")
    all_pass = False

# Check 2: No NaNs
nan_count = np.isnan(emb_loaded).sum()
print(f"\n[2] NaN values: {nan_count}")
if nan_count == 0:
    print("    ✅ PASS")
else:
    print(f"    ❌ FAIL — {nan_count} NaN values found")
    all_pass = False

# Check 3: No zero vectors
zero_rows = (np.abs(emb_loaded).sum(axis=1) == 0).sum()
print(f"\n[3] Zero vectors: {zero_rows}")
if zero_rows == 0:
    print("    ✅ PASS")
else:
    print(f"    ⚠️  {zero_rows} zero vectors found")
    all_pass = False

# Check 4: Value range
vmin = emb_loaded.min()
vmax = emb_loaded.max()
print(f"\n[4] Value range: [{vmin:.3f}, {vmax:.3f}]")
if -5 <= vmin and vmax <= 5:
    print("    ✅ PASS — values in normal range")
else:
    print("    ⚠️  Unusual value range")
    all_pass = False

# Check 5: Similar sentences → similar embeddings
def cosine_sim(a, b):
    return np.dot(a, b) / (norm(a) * norm(b))

sim_text_a = "Half of 48 is 24"
sim_text_b = "48 divided by 2 equals 24"
emb_a = model.encode(sim_text_a)
emb_b = model.encode(sim_text_b)
sim_score = cosine_sim(emb_a, emb_b)
print(f"\n[5] Cosine similarity of similar sentences: {sim_score:.3f}")
print(f"    '{sim_text_a}'")
print(f"    '{sim_text_b}'")
if sim_score > 0.7:
    print("    ✅ PASS — similar sentences have similar embeddings")
else:
    print("    ⚠️  Similar sentences not close enough")
    all_pass = False

# Check 6: Different sentences → different embeddings
diff_text_a = "Half of 48 is 24"
diff_text_b = "She went to the supermarket to buy apples"
emb_c = model.encode(diff_text_a)
emb_d = model.encode(diff_text_b)
diff_score = cosine_sim(emb_c, emb_d)
print(f"\n[6] Cosine similarity of different sentences: {diff_score:.3f}")
print(f"    '{diff_text_a}'")
print(f"    '{diff_text_b}'")
if diff_score < 0.5:
    print("    ✅ PASS — different sentences have different embeddings")
else:
    print("    ⚠️  Different sentences too similar")
    all_pass = False

print()
print("=" * 55)
if all_pass:
    print("  ✅ ALL CHECKS PASSED — Stage 3 complete!")
    print("  ➡️  Ready for Stage 4 (build graphs)")
else:
    print("  ⚠️  SOME CHECKS FAILED — see above")
print("=" * 55)

---
## Stage 4: Build Reasoning Graphs

**Input:** `data/gsm8k_embeddings.npy` + `data/gsm8k_steps_meta.csv`  
**Output:** `data/graphs.pt` — 1,948 PyTorch Geometric graph objects

Each trace becomes a graph. Each step is a node (features = 768-dim embedding). We connect steps using 4 edge types:
- **Type 0 — Sequential:** Step i → Step i+1 (always added, bidirectional)
- **Type 1 — Semantic:** Steps with cosine similarity > 0.75 (similar meaning)
- **Type 2 — Value-reuse:** Number from Step i reappears in Step j (arithmetic dependency)
- **Type 3 — Mentions-number:** A number appears in 3+ steps — all those steps connected (recurring key values)

Edge type is stored as `edge_attr` so GAT can learn different attention per edge type.


In [ ]:
import os
import re
import numpy as np
import pandas as pd
import torch
from torch_geometric.data import Data
import time

DATA_DIR    = os.path.join(os.getcwd(), 'data')
EMB_FILE    = os.path.join(DATA_DIR, 'gsm8k_embeddings.npy')
META_FILE   = os.path.join(DATA_DIR, 'gsm8k_steps_meta.csv')
OUTPUT_FILE = os.path.join(DATA_DIR, 'graphs.pt')

embeddings = np.load(EMB_FILE)
meta       = pd.read_csv(META_FILE)
print(f"Embeddings: {embeddings.shape} | Traces: {meta['trace_id'].nunique()}")


In [ ]:
def sequential_edges(n):
    """
    Build sequential edges: Step i -> Step i+1 in both directions.
    edge_attr type = 0
    """
    edges = []
    for i in range(n - 1):
        edges.append((i, i + 1))
        edges.append((i + 1, i))
    return edges


def cosine_similarity_matrix(embs):
    """Pairwise cosine similarity. Returns (n, n) matrix."""
    norms = np.linalg.norm(embs, axis=1, keepdims=True)
    norms = np.clip(norms, a_min=1e-9, a_max=None)
    normalized = embs / norms
    return normalized @ normalized.T


def semantic_edges(step_embeddings, threshold=0.75):
    """
    Add edges between non-adjacent steps with cosine similarity > threshold.
    edge_attr type = 1
    """
    edges = []
    n = len(step_embeddings)
    if n < 2:
        return edges
    sim_matrix = cosine_similarity_matrix(step_embeddings)
    for i in range(n):
        for j in range(i + 2, n):
            if sim_matrix[i, j] > threshold:
                edges.append((i, j))
                edges.append((j, i))
    return edges


def extract_numbers(text):
    """Extract all numbers > 2 from step text."""
    nums = re.findall(r'\b\d+\.?\d*\b', str(text))
    return {x for x in nums if float(x) > 2}


def value_reuse_edges(step_texts):
    """
    Add edges when a number from Step i reappears in Step j (j > i+1).
    Captures arithmetic dependencies.
    edge_attr type = 2
    """
    edges = []
    n = len(step_texts)
    if n < 2:
        return edges
    step_numbers = [extract_numbers(t) for t in step_texts]
    for i in range(n):
        for j in range(i + 1, n):
            shared = step_numbers[i] & step_numbers[j]
            if shared:
                edges.append((i, j))
                edges.append((j, i))
    return edges


def mentions_number_edges(step_texts):
    """
    NEW — inspired by client's entity node approach.
    For each unique number that appears in 3+ steps,
    connect ALL steps that mention it to each other.

    This differs from value_reuse_edges:
    - value_reuse: only connects step i -> step j (j > i) — directional
    - mentions_number: connects ALL steps mentioning the same key number
      regardless of order — captures global numeric context

    Only fires for numbers appearing in 3+ steps (truly significant numbers).
    edge_attr type = 3
    """
    edges = []
    n = len(step_texts)
    if n < 3:
        return edges

    step_numbers = [extract_numbers(t) for t in step_texts]

    # Find all numbers and which steps mention them
    number_to_steps = {}
    for step_idx, nums in enumerate(step_numbers):
        for num in nums:
            if num not in number_to_steps:
                number_to_steps[num] = []
            number_to_steps[num].append(step_idx)

    # Add edges between ALL steps sharing a number that appears 3+ times
    added = set()
    for num, steps_mentioning in number_to_steps.items():
        if len(steps_mentioning) >= 3:     # only truly recurring numbers
            for a in steps_mentioning:
                for b in steps_mentioning:
                    if a != b:
                        pair = (min(a,b), max(a,b))
                        if pair not in added:
                            edges.append((a, b))
                            edges.append((b, a))
                            added.add(pair)
    return edges


# ── Quick test ────────────────────────────────────────────────────────────
print('TEST — 4-step trace:')
print()
test_texts = [
    'Natalia sold 48 clips in April.',
    'In May she sold half of 48 which is 24 clips.',
    'Total clips = 48 + 24 = 72.',
    'The answer is 72.',
]
test_embs = embeddings[:4]

seq  = sequential_edges(4)
sem  = semantic_edges(test_embs, threshold=0.75)
val  = value_reuse_edges(test_texts)
men  = mentions_number_edges(test_texts)

print(f'Sequential edges:       {seq}')
print(f'Semantic edges:         {sem}')
print(f'Value-reuse edges:      {val}')
print(f'Mentions-number edges:  {men}')
print()
print('Note: 48 appears in steps 0,1,2 -> mentions_number connects all 3')
print()
print('✅ All 4 edge functions working')


In [ ]:
import time

# Edge type legend:
# 0 = sequential      (Step i -> Step i+1)
# 1 = semantic        (cosine similarity > 0.75)
# 2 = value-reuse     (number from step i reappears in step j)
# 3 = mentions-number (same number appears in 3+ steps — all connected)

graphs     = []
skipped    = 0
edge_stats = {'sequential': 0, 'semantic': 0, 'value_reuse': 0, 'mentions_number': 0}

trace_ids = meta['trace_id'].unique()
print(f'Building graphs for {len(trace_ids)} traces...')
print()

start = time.time()

for trace_id in trace_ids:

    trace_rows   = meta[meta['trace_id'] == trace_id].sort_values('step_num')
    step_indices = trace_rows.index.tolist()
    n_steps      = len(step_indices)

    if n_steps < 2:
        skipped += 1
        continue

    node_features = embeddings[step_indices]
    step_texts    = trace_rows['step_text'].tolist()

    # Build all 4 edge types
    seq_edges = sequential_edges(n_steps)
    sem_edges = semantic_edges(node_features, threshold=0.75)
    val_edges = value_reuse_edges(step_texts)
    men_edges = mentions_number_edges(step_texts)

    edge_stats['sequential']      += len(seq_edges)
    edge_stats['semantic']        += len(sem_edges)
    edge_stats['value_reuse']     += len(val_edges)
    edge_stats['mentions_number'] += len(men_edges)

    # Track which type each edge came from (priority: seq > sem > val > men)
    seq_set = set(map(tuple, seq_edges))
    sem_set = set(map(tuple, sem_edges))
    val_set = set(map(tuple, val_edges))
    men_set = set(map(tuple, men_edges))

    all_edge_set = seq_set | sem_set | val_set | men_set
    all_edges    = list(all_edge_set)

    if len(all_edges) == 0:
        skipped += 1
        continue

    # Assign edge type (first match wins: seq > sem > val > men)
    edge_type_list = []
    for e in all_edges:
        et = tuple(e)
        if et in seq_set:
            edge_type_list.append(0)
        elif et in sem_set:
            edge_type_list.append(1)
        elif et in val_set:
            edge_type_list.append(2)
        else:
            edge_type_list.append(3)

    x = torch.tensor(node_features, dtype=torch.float)

    edge_index = torch.tensor(
        [[e[0] for e in all_edges],
         [e[1] for e in all_edges]],
        dtype=torch.long
    )

    edge_attr = torch.tensor(edge_type_list, dtype=torch.float).unsqueeze(1)

    label = int(trace_rows['label'].iloc[0])
    y     = torch.tensor([label], dtype=torch.long)
    split = trace_rows['split'].iloc[0]

    graph = Data(
        x          = x,
        edge_index = edge_index,
        edge_attr  = edge_attr,
        y          = y,
        trace_id   = trace_id,
        split      = split,
        n_steps    = n_steps
    )
    graphs.append(graph)

elapsed = time.time() - start

print(f'Done in {elapsed:.1f} seconds')
print()
print(f'Graphs created:  {len(graphs)}')
print(f'Traces skipped:  {skipped}')
print()
print('Edge counts across all graphs:')
for etype, count in edge_stats.items():
    print(f'  {etype:<20}: {count}')
print()

# Verify edge_attr values are 0-3
sample_attrs = [sorted(g.edge_attr.unique().tolist()) for g in graphs[:3]]
print(f'Sample edge_attr values (first 3 graphs): {sample_attrs}')
print('(should contain values from {0.0, 1.0, 2.0, 3.0})')
print()

train_g = [g for g in graphs if g.split == 'train']
val_g   = [g for g in graphs if g.split == 'val']
test_g  = [g for g in graphs if g.split == 'test']
print(f'Split: Train={len(train_g)}  Val={len(val_g)}  Test={len(test_g)}')


In [ ]:
torch.save(graphs, OUTPUT_FILE)

file_size = os.path.getsize(OUTPUT_FILE) / 1e6
print(f"✅ Saved: {OUTPUT_FILE}")
print(f"   Graphs: {len(graphs)}")
print(f"   Size:   {file_size:.1f} MB")
print()

# Verify we can reload it
reloaded = torch.load(OUTPUT_FILE)
print(f"✅ Reload test passed — {len(reloaded)} graphs loaded back correctly")

In [ ]:
graphs_check = torch.load(OUTPUT_FILE, weights_only=False)

print('=' * 55)
print('  STAGE 4 QUALITY CHECKS')
print('=' * 55)

all_pass = True

# Check 1: Total graphs
print(f'\n[1] Total graphs: {len(graphs_check)}')
if len(graphs_check) >= 280:
    print('    ✅ PASS')
else:
    print('    ❌ FAIL — too few graphs')
    all_pass = False

# Check 2: Node feature shape
bad_shape = sum(1 for g in graphs_check if g.x.shape[1] != 768)
print(f'\n[2] Graphs with wrong node feature shape: {bad_shape}')
if bad_shape == 0:
    print('    ✅ PASS — all nodes have 768 features')
else:
    print('    ❌ FAIL')
    all_pass = False

# Check 3: Edge index valid
bad_edges = sum(1 for g in graphs_check if g.edge_index.max().item() >= g.x.shape[0])
print(f'\n[3] Graphs with out-of-bounds edge indices: {bad_edges}')
if bad_edges == 0:
    print('    ✅ PASS — all edges valid')
else:
    print('    ❌ FAIL')
    all_pass = False

# Check 4: Both labels present
all_labels = [g.y.item() for g in graphs_check]
n_correct  = sum(1 for l in all_labels if l == 1)
n_wrong    = sum(1 for l in all_labels if l == 0)
print(f'\n[4] Label distribution:')
print(f'    Correct (1): {n_correct}')
print(f'    Wrong   (0): {n_wrong}')
if n_correct > 0 and n_wrong > 0:
    print('    ✅ PASS')
else:
    print('    ❌ FAIL')
    all_pass = False

# Check 5: All 3 splits
splits = set(g.split for g in graphs_check)
print(f'\n[5] Splits: {splits}')
for s in ['train', 'val', 'test']:
    print(f'    {s}: {sum(1 for g in graphs_check if g.split == s)}')
if {'train','val','test'}.issubset(splits):
    print('    ✅ PASS')
else:
    print('    ❌ FAIL')
    all_pass = False

# Check 6: Average edges
avg_edges = sum(g.edge_index.shape[1] for g in graphs_check) / len(graphs_check)
avg_nodes = sum(g.x.shape[0] for g in graphs_check) / len(graphs_check)
print(f'\n[6] Avg nodes/graph: {avg_nodes:.1f}  |  Avg edges/graph: {avg_edges:.1f}')
if avg_edges > 2:
    print('    ✅ PASS')
else:
    print('    ⚠️  Very few edges')
    all_pass = False

# Check 7: Edge attr present and values are 0-3 (now 4 types)
missing_attr = sum(1 for g in graphs_check if not hasattr(g,'edge_attr') or g.edge_attr is None)
print(f'\n[7] Graphs with edge_attr: {len(graphs_check)-missing_attr}/{len(graphs_check)}')
if missing_attr == 0:
    valid_types = all(
        set(g.edge_attr.squeeze().tolist()).issubset({0.0,1.0,2.0,3.0})
        for g in graphs_check
    )
    # Count how many graphs use edge type 3
    has_type3 = sum(1 for g in graphs_check if 3.0 in g.edge_attr.squeeze().tolist())
    if valid_types:
        print(f'    ✅ PASS — edge types 0/1/2/3 present')
        print(f'    Graphs using edge type 3 (mentions-number): {has_type3}')
    else:
        print('    ⚠️  Unexpected edge type values')
else:
    print(f'    ❌ FAIL — {missing_attr} graphs missing edge_attr')
    all_pass = False

# Check 8: Edge type distribution across all graphs
from collections import Counter
all_edge_types = []
for g in graphs_check:
    all_edge_types.extend([int(x) for x in g.edge_attr.squeeze().tolist()])
type_counts = Counter(all_edge_types)
total_edges = sum(type_counts.values())
print(f'\n[8] Edge type distribution across all graphs:')
type_names = {0:'sequential', 1:'semantic', 2:'value-reuse', 3:'mentions-number'}
for t in sorted(type_counts.keys()):
    pct = type_counts[t]/total_edges*100
    print(f'    Type {t} ({type_names.get(t,"?")}): {type_counts[t]:>6} ({pct:.1f}%)')
print('    ✅ PASS')

print()
print('=' * 55)
if all_pass:
    print('  ✅ ALL CHECKS PASSED — Stage 4 complete!')
    print('  ➡️  Ready for Stage 5 (train GNN)')
else:
    print('  ⚠️  SOME CHECKS FAILED — see above')
print('=' * 55)


---
## Stage 5: Train GNN Models

**Input:** `data/graphs.pt`  
**Output:** `data/gcn_model.pt`, `data/gat_model.pt`, `data/results.json`

We train and compare three approaches:
1. **Text Baseline** — logistic regression on averaged step embeddings (no graph)
2. **Graph Feature Baseline** — logistic regression on structural graph statistics (no text)
3. **GCN** — Graph Convolutional Network (treats all edges equally)
4. **GAT** — Graph Attention Network (learns different attention per edge type using `edge_dim=1`)

All models trained with **Focal Loss** (α=0.8, γ=3.0) to handle 75/25 class imbalance. Each model trained 3× with different seeds; results averaged.


In [ ]:
import os
import json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.loader import DataLoader as GeoDataLoader
from torch_geometric.nn import GCNConv, GATConv, global_mean_pool
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

DATA_DIR     = os.path.join(os.getcwd(), 'data')
GRAPHS_FILE  = os.path.join(DATA_DIR, 'graphs.pt')
GCN_MODEL    = os.path.join(DATA_DIR, 'gcn_model.pt')
GAT_MODEL    = os.path.join(DATA_DIR, 'gat_model.pt')
RESULTS_FILE = os.path.join(DATA_DIR, 'results.json')
device       = torch.device('cpu')

all_graphs   = torch.load(GRAPHS_FILE, weights_only=False)
train_graphs = [g for g in all_graphs if g.split == 'train']
val_graphs   = [g for g in all_graphs if g.split == 'val']
test_graphs  = [g for g in all_graphs if g.split == 'test']
print(f"Train: {len(train_graphs)} | Val: {len(val_graphs)} | Test: {len(test_graphs)}")
print(f"Wrong in train: {sum(1 for g in train_graphs if g.y.item()==0)}")


In [ ]:
# ── GCN Model — does NOT use edge_attr (GCNConv ignores edge features) ──────
class GCNModel(nn.Module):
    def __init__(self, input_dim=768, hidden_dim=128, out_dim=64, num_classes=2, dropout=0.3):
        super().__init__()
        self.conv1   = GCNConv(input_dim, hidden_dim)
        self.conv2   = GCNConv(hidden_dim, out_dim)
        self.mlp     = nn.Sequential(
            nn.Linear(out_dim, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, num_classes)
        )
        self.dropout = dropout

    def forward(self, x, edge_index, batch, edge_attr=None):
        # GCNConv does not use edge_attr — included for consistent interface
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.conv2(x, edge_index))
        x = global_mean_pool(x, batch)
        return self.mlp(x)


# ── GAT Model — USES edge_attr (edge_dim=1 for edge type 0/1/2) ─────────────
class GATModel(nn.Module):
    def __init__(self, input_dim=768, hidden_dim=128, out_dim=64, num_classes=2, heads=4, dropout=0.3):
        super().__init__()
        # edge_dim=1 tells GATConv to use the edge type feature
        self.conv1   = GATConv(input_dim, hidden_dim // heads, heads=heads,
                               edge_dim=1, dropout=dropout)
        self.conv2   = GATConv(hidden_dim, out_dim, heads=1,
                               concat=False, edge_dim=1, dropout=dropout)
        self.mlp     = nn.Sequential(
            nn.Linear(out_dim, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, num_classes)
        )
        self.dropout = dropout

    def forward(self, x, edge_index, batch, edge_attr=None):
        # GAT uses edge_attr to learn different attention per edge type
        x = F.relu(self.conv1(x, edge_index, edge_attr=edge_attr))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.conv2(x, edge_index, edge_attr=edge_attr))
        x = global_mean_pool(x, batch)
        return self.mlp(x)


# ── Quick test ────────────────────────────────────────────────────────────────
sample    = train_graphs[0]
batch_vec = torch.zeros(sample.x.shape[0], dtype=torch.long)
edge_attr = sample.edge_attr if hasattr(sample, 'edge_attr') and sample.edge_attr is not None else None

gcn_test = GCNModel()
gat_test = GATModel()

out_gcn = gcn_test(sample.x, sample.edge_index, batch_vec, edge_attr)
out_gat = gat_test(sample.x, sample.edge_index, batch_vec, edge_attr)

print(f"GCN output shape: {out_gcn.shape}  (1 graph, 2 class scores)")
print(f"GAT output shape: {out_gat.shape}  (1 graph, 2 class scores)")
print()
print(f"GCN parameters: {sum(p.numel() for p in gcn_test.parameters()):,}")
print(f"GAT parameters: {sum(p.numel() for p in gat_test.parameters()):,}")
print()
has_edge_attr = edge_attr is not None
print(f"Edge attributes available: {has_edge_attr}")
if has_edge_attr:
    print(f"Edge attr shape: {edge_attr.shape}  (one value per edge)")
    print(f"Edge types seen: {edge_attr.unique().tolist()}  (0=sequential, 1=semantic, 2=value-reuse)")
print()
print("✅ Both models working correctly")

In [ ]:
from torch_geometric.loader import DataLoader as GeoDataLoader
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, average_precision_score

# Per-class alpha focal loss (proper form; minority class gets alpha, majority gets 1-alpha).
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.75, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, reduction='none')
        pt = torch.exp(-ce)
        alpha_t = torch.where(targets == 0,
                              torch.full_like(ce, self.alpha),
                              torch.full_like(ce, 1.0 - self.alpha))
        return (alpha_t * (1.0 - pt) ** self.gamma * ce).mean()


def train_epoch(model, graphs, optimizer, criterion):
    model.train()
    total = 0
    loader = GeoDataLoader(graphs, batch_size=16, shuffle=True)
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        out  = model(batch.x, batch.edge_index, batch.batch,
                     getattr(batch, 'edge_attr', None))
        loss = criterion(out, batch.y)
        loss.backward(); optimizer.step()
        total += loss.item()
    return total / max(len(loader), 1)


def evaluate(model, graphs, criterion):
    """Default-threshold (argmax) evaluation. Returns loss, acc, f1, auc_roc, pr_auc."""
    model.eval()
    preds_all, probs_all, labels_all = [], [], []
    total = 0
    loader = GeoDataLoader(graphs, batch_size=16, shuffle=False)
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            out = model(batch.x, batch.edge_index, batch.batch,
                        getattr(batch, 'edge_attr', None))
            total += criterion(out, batch.y).item()
            probs_correct = F.softmax(out, dim=1)[:, 1]
            preds_all.extend(out.argmax(dim=1).cpu().numpy())
            probs_all.extend(probs_correct.cpu().numpy())
            labels_all.extend(batch.y.cpu().numpy())
    acc = accuracy_score(labels_all, preds_all)
    f1  = f1_score(labels_all, preds_all, average='weighted', zero_division=0)
    try:
        auc = roc_auc_score(labels_all, probs_all)
    except Exception:
        auc = float('nan')
    try:
        pr_auc = average_precision_score(1 - np.array(labels_all),
                                         1 - np.array(probs_all))
    except Exception:
        pr_auc = float('nan')
    return total / max(len(loader), 1), acc, f1, auc, pr_auc


def train_model(model_class, train_g, val_g, test_g, seed=42, epochs=200, lr=5e-4, patience=30):
    torch.manual_seed(seed); np.random.seed(seed)
    model     = model_class().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    criterion = FocalLoss(alpha=0.75, gamma=2.0)

    best_val_loss, best_state, patience_count = float('inf'), None, 0
    for epoch in range(1, epochs + 1):
        tr = train_epoch(model, train_g, optimizer, criterion)
        v_loss, v_acc, v_f1, v_auc, v_pr = evaluate(model, val_g, criterion)
        if v_loss < best_val_loss:
            best_val_loss = v_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            patience_count = 0
        else:
            patience_count += 1
        if epoch % 10 == 0:
            print(f'  ep{epoch:3d} train={tr:.4f} val={v_loss:.4f} acc={v_acc:.3f} f1={v_f1:.3f} auc={v_auc:.3f} pr={v_pr:.3f}')
        if patience_count >= patience:
            print(f'  Early stop at epoch {epoch}'); break
    model.load_state_dict(best_state)
    _, t_acc, t_f1, t_auc, t_pr = evaluate(model, test_g, criterion)
    return model, t_acc, t_f1, t_auc, t_pr


print('FocalLoss (per-class alpha) + evaluation (argmax / 0.5 threshold) defined.')
print('Reports: Accuracy, weighted F1, AUC-ROC, PR-AUC.')


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

def graphs_to_flat_features(graphs):
    X = np.array([g.x.numpy().mean(axis=0) for g in graphs])
    y = np.array([g.y.item() for g in graphs])
    return X, y

print('Training text baseline...\n')
baseline_accs, baseline_f1s, baseline_aucs, baseline_prs = [], [], [], []
for seed in [42, 123, 456]:
    X_tr_, y_tr_ = graphs_to_flat_features(train_graphs)
    X_v,   y_v   = graphs_to_flat_features(val_graphs)
    X_t,   y_t   = graphs_to_flat_features(test_graphs)
    X_tr = np.vstack([X_tr_, X_v]); y_tr = np.concatenate([y_tr_, y_v])
    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr); X_t_s = scaler.transform(X_t)
    clf = LogisticRegression(max_iter=1000, random_state=seed, class_weight='balanced')
    clf.fit(X_tr_s, y_tr)
    preds = clf.predict(X_t_s); probs = clf.predict_proba(X_t_s)[:, 1]
    acc = accuracy_score(y_t, preds)
    f1  = f1_score(y_t, preds, average='weighted', zero_division=0)
    auc = roc_auc_score(y_t, probs) if len(set(y_t)) > 1 else float('nan')
    pr  = average_precision_score(1 - y_t, 1 - probs) if len(set(y_t)) > 1 else float('nan')
    baseline_accs.append(acc); baseline_f1s.append(f1); baseline_aucs.append(auc); baseline_prs.append(pr)
    print(f'  seed {seed}: acc={acc:.3f} f1={f1:.3f} auc={auc:.3f} pr={pr:.3f}')

print('\nText Baseline (mean ± std over 3 seeds):')
print(f'  Acc={np.mean(baseline_accs):.3f}±{np.std(baseline_accs):.3f}  F1={np.mean(baseline_f1s):.3f}  AUC={np.mean(baseline_aucs):.3f}  PR={np.mean(baseline_prs):.3f}')


In [ ]:
def graph_to_feature_vector(g):
    n_nodes = g.x.shape[0]; n_edges = g.edge_index.shape[1]
    avg_deg = n_edges / max(n_nodes, 1)
    if g.edge_attr is not None and g.edge_attr.shape[0] > 0:
        attrs = g.edge_attr.squeeze().tolist()
        if isinstance(attrs, float): attrs = [attrs]
        T = max(len(attrs), 1)
        p_seq = sum(1 for a in attrs if int(a)==0)/T
        p_sem = sum(1 for a in attrs if int(a)==1)/T
        p_val = sum(1 for a in attrs if int(a)==2)/T
        p_men = sum(1 for a in attrs if int(a)==3)/T
    else:
        p_seq=p_sem=p_val=p_men=0.0
    density = n_edges / max(n_nodes*(n_nodes-1), 1)
    return np.array([n_nodes, n_edges, avg_deg, p_seq, p_sem, p_val, p_men, density], dtype=np.float32)

def train_graph_feature_baseline(train_g, val_g, test_g, seed=42):
    np.random.seed(seed)
    X_tr_ = np.array([graph_to_feature_vector(g) for g in train_g])
    X_v   = np.array([graph_to_feature_vector(g) for g in val_g])
    X_t   = np.array([graph_to_feature_vector(g) for g in test_g])
    y_tr_ = np.array([g.y.item() for g in train_g])
    y_v   = np.array([g.y.item() for g in val_g])
    y_t   = np.array([g.y.item() for g in test_g])
    X_tr  = np.vstack([X_tr_, X_v]); y_tr = np.concatenate([y_tr_, y_v])
    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr); X_t_s = scaler.transform(X_t)
    clf = LogisticRegression(max_iter=1000, random_state=seed, class_weight='balanced')
    clf.fit(X_tr_s, y_tr)
    preds = clf.predict(X_t_s); probs = clf.predict_proba(X_t_s)[:, 1]
    acc = accuracy_score(y_t, preds)
    f1  = f1_score(y_t, preds, average='weighted', zero_division=0)
    auc = roc_auc_score(y_t, probs) if len(set(y_t)) > 1 else float('nan')
    pr  = average_precision_score(1 - y_t, 1 - probs) if len(set(y_t)) > 1 else float('nan')
    return acc, f1, auc, pr

print('Training graph feature baseline...\n')
gf_accs, gf_f1s, gf_aucs, gf_prs = [], [], [], []
for seed in [42, 123, 456]:
    acc, f1, auc, pr = train_graph_feature_baseline(train_graphs, val_graphs, test_graphs, seed=seed)
    gf_accs.append(acc); gf_f1s.append(f1); gf_aucs.append(auc); gf_prs.append(pr)
    print(f'  seed {seed}: acc={acc:.3f} f1={f1:.3f} auc={auc:.3f} pr={pr:.3f}')
print('\nGraph Feature Baseline (mean ± std over 3 seeds):')
print(f'  Acc={np.mean(gf_accs):.3f}±{np.std(gf_accs):.3f}  F1={np.mean(gf_f1s):.3f}  AUC={np.mean(gf_aucs):.3f}  PR={np.mean(gf_prs):.3f}')


In [ ]:
import time
print('Training GCN (3 seeds)...\n' + '='*60)
gcn_accs, gcn_f1s, gcn_aucs, gcn_prs = [], [], [], []
best_gcn, best_gcn_auc = None, -1
t0 = time.time()
for seed in [42, 123, 456]:
    print(f'\nSeed {seed}:')
    model, acc, f1, auc, pr = train_model(GCNModel, train_graphs, val_graphs, test_graphs,
                                          seed=seed, epochs=200, lr=5e-4, patience=30)
    gcn_accs.append(acc); gcn_f1s.append(f1); gcn_aucs.append(auc); gcn_prs.append(pr)
    print(f'  Test: acc={acc:.3f} f1={f1:.3f} auc={auc:.3f} pr={pr:.3f}')
    if auc > best_gcn_auc:
        best_gcn_auc, best_gcn = auc, model
print('\n' + '='*60)
print(f'GCN: Acc={np.mean(gcn_accs):.3f}±{np.std(gcn_accs):.3f}  F1={np.mean(gcn_f1s):.3f}  AUC={np.mean(gcn_aucs):.3f}  PR={np.mean(gcn_prs):.3f}')
print(f'Time: {(time.time()-t0)/60:.1f} min')
torch.save(best_gcn.state_dict(), GCN_MODEL)
print(f'Best GCN (by AUC) saved: {GCN_MODEL}')


In [ ]:
print('Training GAT (3 seeds)...\n' + '='*60)
gat_accs, gat_f1s, gat_aucs, gat_prs = [], [], [], []
best_gat, best_gat_auc = None, -1
t0 = time.time()
for seed in [42, 123, 456]:
    print(f'\nSeed {seed}:')
    model, acc, f1, auc, pr = train_model(GATModel, train_graphs, val_graphs, test_graphs,
                                          seed=seed, epochs=200, lr=5e-4, patience=30)
    gat_accs.append(acc); gat_f1s.append(f1); gat_aucs.append(auc); gat_prs.append(pr)
    print(f'  Test: acc={acc:.3f} f1={f1:.3f} auc={auc:.3f} pr={pr:.3f}')
    if auc > best_gat_auc:
        best_gat_auc, best_gat = auc, model
print('\n' + '='*60)
print(f'GAT: Acc={np.mean(gat_accs):.3f}±{np.std(gat_accs):.3f}  F1={np.mean(gat_f1s):.3f}  AUC={np.mean(gat_aucs):.3f}  PR={np.mean(gat_prs):.3f}')
print(f'Time: {(time.time()-t0)/60:.1f} min')
torch.save(best_gat.state_dict(), GAT_MODEL)
print(f'Best GAT (by AUC) saved: {GAT_MODEL}')


In [ ]:
from scipy import stats as sp_stats

print('=' * 78)
print('  FINAL RESULTS (default 0.5 threshold; mean over 3 seeds)')
print('=' * 78)
print(f"{'Model':<28} {'Accuracy':>10} {'F1':>10} {'AUC-ROC':>10} {'PR-AUC':>10}")
print('-' * 78)
print(f"{'Text Baseline':<28} {np.mean(baseline_accs):>10.3f} {np.mean(baseline_f1s):>10.3f} {np.mean(baseline_aucs):>10.3f} {np.mean(baseline_prs):>10.3f}")
print(f"{'Graph Feature Baseline':<28} {np.mean(gf_accs):>10.3f} {np.mean(gf_f1s):>10.3f} {np.mean(gf_aucs):>10.3f} {np.mean(gf_prs):>10.3f}")
print(f"{'GCN':<28} {np.mean(gcn_accs):>10.3f} {np.mean(gcn_f1s):>10.3f} {np.mean(gcn_aucs):>10.3f} {np.mean(gcn_prs):>10.3f}")
print(f"{'GAT':<28} {np.mean(gat_accs):>10.3f} {np.mean(gat_f1s):>10.3f} {np.mean(gat_aucs):>10.3f} {np.mean(gat_prs):>10.3f}")
print('=' * 78)
print()
def paired_p(a, b):
    try:
        _, p = sp_stats.ttest_rel(a, b); return p
    except Exception:
        return float('nan')
print('Paired t-tests across 3 seeds (low power; directional only):')
print(f'  GCN vs Text Baseline (AUC-ROC):           p = {paired_p(gcn_aucs, baseline_aucs):.3f}')
print(f'  GCN vs Graph Feature Baseline (AUC-ROC):  p = {paired_p(gcn_aucs, gf_aucs):.3f}')
print(f'  GAT vs Graph Feature Baseline (AUC-ROC):  p = {paired_p(gat_aucs, gf_aucs):.3f}')
print(f'  GAT vs GCN (AUC-ROC):                     p = {paired_p(gat_aucs, gcn_aucs):.3f}')

def pack(accs, f1s, aucs, prs):
    return {
        'accuracy': float(np.mean(accs)), 'accuracy_std': float(np.std(accs)),
        'f1': float(np.mean(f1s)), 'f1_std': float(np.std(f1s)),
        'auc_roc': float(np.mean(aucs)), 'auc_roc_std': float(np.std(aucs)),
        'pr_auc': float(np.mean(prs)), 'pr_auc_std': float(np.std(prs)),
        'per_seed': {'accuracy': list(map(float, accs)), 'f1': list(map(float, f1s)),
                     'auc_roc': list(map(float, aucs)), 'pr_auc': list(map(float, prs))}
    }

results = {
    'evaluation_protocol': {
        'threshold': 'default (argmax / 0.5)',
        'seeds': [42, 123, 456],
        'metrics': ['accuracy', 'f1_weighted', 'auc_roc', 'pr_auc']
    },
    'text_baseline':          pack(baseline_accs, baseline_f1s, baseline_aucs, baseline_prs),
    'graph_feature_baseline': pack(gf_accs, gf_f1s, gf_aucs, gf_prs),
    'gcn':                    pack(gcn_accs, gcn_f1s, gcn_aucs, gcn_prs),
    'gat':                    pack(gat_accs, gat_f1s, gat_aucs, gat_prs),
}
with open(RESULTS_FILE, 'w') as f:
    json.dump(results, f, indent=2)
print(f'\nResults saved: {RESULTS_FILE}')


---
## Stage 6: Evaluation Report and Charts

**Input:** `data/results.json` + saved model files  
**Output:** `data/results_chart.png`, `data/confusion_matrix.png`, `data/evaluation_report.txt`

No training here — we just read the Stage 5 results and visualise them. Three outputs:
1. Bar chart comparing all models across Accuracy, F1, AUC-ROC
2. Confusion matrices for GCN and GAT on the test set
3. Seed consistency chart showing stability across 3 runs
4. Plain text evaluation report


In [ ]:
import os
import json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from torch_geometric.loader import DataLoader as GeoDataLoader
from torch_geometric.nn import GCNConv, GATConv, global_mean_pool
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix, classification_report

matplotlib.rcParams['figure.dpi'] = 120

DATA_DIR      = os.path.join(os.getcwd(), 'data')
GRAPHS_FILE   = os.path.join(DATA_DIR, 'graphs.pt')
GCN_MODEL     = os.path.join(DATA_DIR, 'gcn_model.pt')
GAT_MODEL     = os.path.join(DATA_DIR, 'gat_model.pt')
RESULTS_FILE  = os.path.join(DATA_DIR, 'results.json')
CHART_FILE    = os.path.join(DATA_DIR, 'results_chart.png')
CM_FILE       = os.path.join(DATA_DIR, 'confusion_matrix.png')
REPORT_FILE   = os.path.join(DATA_DIR, 'evaluation_report.txt')
device        = torch.device('cpu')

with open(RESULTS_FILE) as f:
    results = json.load(f)
print("Results loaded successfully")


In [ ]:
with open(RESULTS_FILE) as f:
    results = json.load(f)

# Build model dict — include graph feature baseline if present
models = {'Text Baseline': results['text_baseline']}
if 'graph_feature_baseline' in results:
    models['Graph Feature'] = results['graph_feature_baseline']
models['GCN'] = results['gcn']
models['GAT'] = results['gat']

print('=' * 70)
print('  FINAL RESULTS — Reasoning-as-Graphs')
print('=' * 70)
print(f"{'Model':<26} {'Accuracy':>12} {'F1 Score':>12} {'AUC-ROC':>12}")
print('-' * 65)
for name, m in models.items():
    print(f"{name:<26} {m['accuracy']:>11.1%} {m['f1']:>12.3f} {m['auc_roc']:>12.3f}")
print('=' * 70)
print()

# Improvement over text baseline
base_acc = results['text_baseline']['accuracy']
gcn_acc  = results['gcn']['accuracy']
gat_acc  = results['gat']['accuracy']
print(f"GCN vs Text Baseline: +{(gcn_acc-base_acc)*100:.1f}pp accuracy")
print(f"GAT vs Text Baseline: +{(gat_acc-base_acc)*100:.1f}pp accuracy")
if 'graph_feature_baseline' in results:
    gf_acc = results['graph_feature_baseline']['accuracy']
    print(f"GCN vs Graph Features: +{(gcn_acc-gf_acc)*100:.1f}pp accuracy")
    print(f"GAT vs Graph Features: +{(gat_acc-gf_acc)*100:.1f}pp accuracy")
print()

# Per-seed breakdown
print('Per-seed results (consistency check):')
header = f"{'Model':<26} {'Seed 42':>10} {'Seed 123':>10} {'Seed 456':>10} {'Std':>8}"
print(header)
print('-' * 65)
for name, m in models.items():
    accs = m['per_seed']['accuracy']
    std  = float(np.std(accs))
    print(f"{name:<26} {accs[0]:>10.1%} {accs[1]:>10.1%} {accs[2]:>10.1%} {std:>8.3f}")
print()
print('Low std = consistent results across runs (good)')


In [ ]:
# Build model list dynamically
model_keys = ['text_baseline']
model_labels = ['Text Baseline']
if 'graph_feature_baseline' in results:
    model_keys.append('graph_feature_baseline')
    model_labels.append('Graph Feature')
model_keys.extend(['gcn', 'gat'])
model_labels.extend(['GCN', 'GAT'])

metrics = ['Accuracy', 'F1 Score', 'AUC-ROC']
keys    = ['accuracy', 'f1', 'auc_roc']
colors  = ['#95a5a6', '#7fb3d3', '#2e86c1', '#1a5276'][:len(model_keys)]

vals = {}
stds = {}
for key, name in zip(keys, metrics):
    vals[name] = [results[mk][key] for mk in model_keys]
    stds[name] = [results[mk][key+'_std'] for mk in model_keys]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Reasoning-as-Graphs — Model Comparison', fontsize=14, fontweight='bold', y=1.02)

for ax, metric in zip(axes, metrics):
    bars = ax.bar(model_labels, vals[metric], color=colors,
                  yerr=stds[metric], capsize=5, edgecolor='white', width=0.5)
    for bar, val in zip(bars, vals[metric]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax.set_title(metric, fontsize=12, fontweight='bold')
    ax.set_ylim(0, 1.1)
    ax.set_ylabel('Score')
    ax.tick_params(axis='x', rotation=20)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='y', alpha=0.3)
    ax.axhline(y=vals[metric][0], color='#95a5a6', linestyle='--', alpha=0.5, linewidth=1)

plt.tight_layout()
plt.savefig(CHART_FILE, bbox_inches='tight', dpi=150)
plt.show()
print(f'Chart saved: {CHART_FILE}')


In [ ]:
# Model definitions must match Stage 5 exactly
class GCNModel(nn.Module):
    def __init__(self, input_dim=768, hidden_dim=128, out_dim=64, num_classes=2, dropout=0.3):
        super().__init__()
        self.conv1   = GCNConv(input_dim, hidden_dim)
        self.conv2   = GCNConv(hidden_dim, out_dim)
        self.mlp     = nn.Sequential(
            nn.Linear(out_dim, 32), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(32, num_classes)
        )
        self.dropout = dropout

    def forward(self, x, edge_index, batch, edge_attr=None):
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.conv2(x, edge_index))
        x = global_mean_pool(x, batch)
        return self.mlp(x)


class GATModel(nn.Module):
    def __init__(self, input_dim=768, hidden_dim=128, out_dim=64, num_classes=2, heads=4, dropout=0.3):
        super().__init__()
        self.conv1   = GATConv(input_dim, hidden_dim // heads, heads=heads,
                               edge_dim=1, dropout=dropout)
        self.conv2   = GATConv(hidden_dim, out_dim, heads=1,
                               concat=False, edge_dim=1, dropout=dropout)
        self.mlp     = nn.Sequential(
            nn.Linear(out_dim, 32), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(32, num_classes)
        )
        self.dropout = dropout

    def forward(self, x, edge_index, batch, edge_attr=None):
        x = F.relu(self.conv1(x, edge_index, edge_attr=edge_attr))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.conv2(x, edge_index, edge_attr=edge_attr))
        x = global_mean_pool(x, batch)
        return self.mlp(x)


def get_predictions(model, graphs):
    model.eval()
    all_preds, all_probs, all_labels = [], [], []
    loader = GeoDataLoader(graphs, batch_size=16, shuffle=False)
    with torch.no_grad():
        for batch in loader:
            batch     = batch.to(device)
            edge_attr = batch.edge_attr if hasattr(batch, 'edge_attr') and batch.edge_attr is not None else None
            out       = model(batch.x, batch.edge_index, batch.batch, edge_attr)
            probs     = F.softmax(out, dim=1)[:, 1]
            preds     = out.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(batch.y.cpu().numpy())
    return np.array(all_labels), np.array(all_preds), np.array(all_probs)


# Load graphs
all_graphs  = torch.load(GRAPHS_FILE, weights_only=False)
test_graphs = [g for g in all_graphs if g.split == 'test']

gcn = GCNModel().to(device)
gcn.load_state_dict(torch.load(GCN_MODEL, weights_only=True))
gcn_labels, gcn_preds, gcn_probs = get_predictions(gcn, test_graphs)
print('✅ GCN loaded — test set predictions done')

gat = GATModel().to(device)
gat.load_state_dict(torch.load(GAT_MODEL, weights_only=True))
gat_labels, gat_preds, gat_probs = get_predictions(gat, test_graphs)
print('✅ GAT loaded — test set predictions done')
print()
print(f'Test set size: {len(test_graphs)} graphs')
print(f'  Correct (1): {sum(gcn_labels==1)}')
print(f'  Wrong   (0): {sum(gcn_labels==0)}')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Confusion Matrices — Test Set', fontsize=13, fontweight='bold')

for ax, labels, preds, name, color in [
    (axes[0], gcn_labels, gcn_preds, 'GCN', 'Blues'),
    (axes[1], gat_labels, gat_preds, 'GAT', 'Greens')
]:
    cm = confusion_matrix(labels, preds)
    sns.heatmap(
        cm, annot=True, fmt='d', cmap=color, ax=ax,
        xticklabels=['Predicted\nCorrect', 'Predicted\nWrong'],
        yticklabels=['Actual\nCorrect', 'Actual\nWrong'],
        linewidths=1, linecolor='white', annot_kws={'size': 14}
    )
    acc = accuracy_score(labels, preds)
    f1  = f1_score(labels, preds, average='weighted', zero_division=0)
    ax.set_title(f'{name}\nAccuracy: {acc:.1%}  F1: {f1:.3f}', fontsize=11, fontweight='bold')
    ax.set_ylabel('Actual Label', fontsize=10)
    ax.set_xlabel('Predicted Label', fontsize=10)

plt.tight_layout()
plt.savefig(CM_FILE, bbox_inches='tight', dpi=150)
plt.show()
print(f"✅ Confusion matrix saved: {CM_FILE}")
print()

# Print detailed classification report
print("GCN Classification Report:")
print(classification_report(gcn_labels, gcn_preds,
      target_names=['Correct (1)', 'Wrong (0)'], zero_division=0))
print()
print("GAT Classification Report:")
print(classification_report(gat_labels, gat_preds,
      target_names=['Correct (1)', 'Wrong (0)'], zero_division=0))

In [ ]:
seeds = ['Seed 42', 'Seed 123', 'Seed 456']

# Build model list dynamically
plot_models = [
    ('Text Baseline', 'text_baseline', '#95a5a6'),
]
if 'graph_feature_baseline' in results:
    plot_models.append(('Graph Feature', 'graph_feature_baseline', '#7fb3d3'))
plot_models.extend([
    ('GCN', 'gcn', '#2e86c1'),
    ('GAT', 'gat', '#1a5276'),
])

fig, ax = plt.subplots(figsize=(10, 5))

x      = np.arange(len(seeds))
n_models = len(plot_models)
width  = 0.8 / n_models
offset = -(n_models - 1) * width / 2

for name, key, color in plot_models:
    accs = results[key]['per_seed']['accuracy']
    bars = ax.bar(x + offset, accs, width, label=name, color=color, edgecolor='white')
    for bar, val in zip(bars, accs):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{val:.2f}', ha='center', va='bottom', fontsize=7)
    offset += width

ax.set_title('Accuracy per Seed — Consistency Check', fontsize=12, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(seeds)
ax.set_ylabel('Accuracy')
ax.set_ylim(0, 1.1)
ax.legend()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
seed_chart = os.path.join(DATA_DIR, 'seed_consistency.png')
plt.savefig(seed_chart, bbox_inches='tight', dpi=150)
plt.show()
print(f'Seed consistency chart saved: {seed_chart}')


In [ ]:
gcn_acc  = results['gcn']['accuracy']
gat_acc  = results['gat']['accuracy']
gcn_f1   = results['gcn']['f1']
gat_f1   = results['gat']['f1']
gcn_auc  = results['gcn']['auc_roc']
gat_auc  = results['gat']['auc_roc']
base_acc = results['text_baseline']['accuracy']
base_f1  = results['text_baseline']['f1']
base_auc = results['text_baseline']['auc_roc']
gf_acc   = results.get('graph_feature_baseline', {}).get('accuracy', 0.0)
gf_f1    = results.get('graph_feature_baseline', {}).get('f1', 0.0)
gf_auc   = results.get('graph_feature_baseline', {}).get('auc_roc', 0.0)

# Load graphs dynamically
all_graphs_rep = torch.load(GRAPHS_FILE, weights_only=False)
n_total  = len(all_graphs_rep)
n_train  = sum(1 for g in all_graphs_rep if g.split=='train')
n_val    = sum(1 for g in all_graphs_rep if g.split=='val')
n_test   = sum(1 for g in all_graphs_rep if g.split=='test')
avg_nodes = sum(g.x.shape[0] for g in all_graphs_rep) / n_total
avg_edges = sum(g.edge_index.shape[1] for g in all_graphs_rep) / n_total

lines = [
    '',
    '====================================================================',
    'REASONING-AS-GRAPHS -- EVALUATION REPORT',
    'Machine Learning with Graphs | Spring 2026',
    '====================================================================',
    '',
    'PROJECT OVERVIEW',
    '----------------',
    'We built a system that converts LLM reasoning traces into graphs',
    'and trains a GNN to classify whether the reasoning is correct or wrong.',
    f'Dataset: GSM8K ({n_total} traces, 75% correct / 25% wrong)',
    'LLM: llama-3.1-8b-instant via Groq API (temperature=0.6)',
    'Node features: 768-dim sentence embeddings (all-mpnet-base-v2)',
    'Edge types: Sequential(0), Semantic(1), Value-reuse(2), Mentions-number(3)',
    '',
    'RESULTS SUMMARY',
    '---------------',
    f"{'Model':<26} {'Accuracy':>10} {'F1':>8} {'AUC-ROC':>10}",
    f"{'Text Baseline':<26} {base_acc:>10.1%} {base_f1:>8.3f} {base_auc:>10.3f}",
    f"{'Graph Feature Baseline':<26} {gf_acc:>10.1%} {gf_f1:>8.3f} {gf_auc:>10.3f}",
    f"{'GCN':<26} {gcn_acc:>10.1%} {gcn_f1:>8.3f} {gcn_auc:>10.3f}",
    f"{'GAT':<26} {gat_acc:>10.1%} {gat_f1:>8.3f} {gat_auc:>10.3f}",
    '',
    'IMPROVEMENT OVER TEXT BASELINE',
    '-------------------------------',
    f'Graph Feature Baseline: {(gf_acc-base_acc)*100:+.1f}pp accuracy',
    f'GCN:                    {(gcn_acc-base_acc)*100:+.1f}pp accuracy',
    f'GAT:                    {(gat_acc-base_acc)*100:+.1f}pp accuracy',
    '',
    'GNN vs GRAPH FEATURE BASELINE',
    '------------------------------',
    f'GCN over graph features: {(gcn_acc-gf_acc)*100:+.1f}pp',
    f'GAT over graph features: {(gat_acc-gf_acc)*100:+.1f}pp',
    '(positive = neural graph learning adds value beyond simple graph stats)',
    '',
    'KEY FINDINGS',
    '------------',
    '1. Both GCN and GAT outperform the text baseline.',
    '2. GNN outperforms the graph feature baseline, confirming neural',
    '   graph learning adds value beyond simple structural statistics.',
    '3. GAT with edge type features learns per-edge-type attention weights.',
    '4. Results consistent across 3 seeds -- improvement is reliable.',
    '',
    'DATASET DETAILS',
    '---------------',
    f'Total traces:  {n_total}',
    f'Train:         {n_train} graphs',
    f'Validation:    {n_val} graphs',
    f'Test:          {n_test} graphs',
    f'Avg steps:     {avg_nodes:.1f} per trace',
    f'Avg edges:     {avg_edges:.1f} per graph',
    '',
    'MODEL DETAILS',
    '-------------',
    'GCN: GCNConv(768->128) + GCNConv(128->64) + GlobalMeanPool + MLP(64->32->2)',
    'GAT: GATConv(768->128,heads=4,edge_dim=1) + GATConv(128->64) + MLP',
    'Loss: Focal Loss (alpha=0.75, gamma=2.0)',
    'Optimizer: Adam lr=0.0005, weight_decay=1e-4, early stopping patience=30',
    'Seeds: [42, 123, 456] -- results averaged over 3 runs',
    '',
    '====================================================================',
    'Next step: Stage 7 -- GNNExplainer to identify which step failed',
    '====================================================================',
    '',
]
report = '\n'.join(lines)
print(report)

with open(REPORT_FILE, 'w', encoding='utf-8') as f:
    f.write(report)
print(f'Report saved: {REPORT_FILE}')


---
## Stage 7: GNNExplainer — Find the Failing Step

**Input:** `data/graphs.pt` + `data/gcn_model.pt` + `data/gsm8k_steps_meta.csv`  
**Output:** `data/explanations.json`, `data/explanation_chart.png`

GNNExplainer asks: "which node (step) was most responsible for this prediction?" It learns a mask over nodes and edges — the node with the highest mask value is the step most responsible for the reasoning failure.

We run it on all 75 wrong traces in the test set, regardless of whether the model caught them. This gives genuine step-level attribution for real LLM failures — with zero human annotation.


In [ ]:
import os
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib
from torch_geometric.nn import GCNConv, GATConv, global_mean_pool
from torch_geometric.explain import Explainer, GNNExplainer
from collections import Counter

matplotlib.rcParams['figure.dpi'] = 120
device = torch.device('cpu')

DATA_DIR    = os.path.join(os.getcwd(), 'data')
GRAPHS_FILE = os.path.join(DATA_DIR, 'graphs.pt')
GCN_MODEL   = os.path.join(DATA_DIR, 'gcn_model.pt')
META_FILE   = os.path.join(DATA_DIR, 'gsm8k_steps_meta.csv')
EXPL_FILE   = os.path.join(DATA_DIR, 'explanations.json')
CHART_FILE  = os.path.join(DATA_DIR, 'explanation_chart.png')


In [ ]:
# GCN model — must match Stage 5 exactly
class GCNModel(nn.Module):
    def __init__(self, input_dim=768, hidden_dim=128, out_dim=64, num_classes=2, dropout=0.3):
        super().__init__()
        self.conv1   = GCNConv(input_dim, hidden_dim)
        self.conv2   = GCNConv(hidden_dim, out_dim)
        self.mlp     = nn.Sequential(
            nn.Linear(out_dim, 32), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(32, num_classes)
        )
        self.dropout = dropout

    def forward(self, x, edge_index, batch=None, edge_attr=None):
        if batch is None:
            batch = torch.zeros(x.shape[0], dtype=torch.long)
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.conv2(x, edge_index))
        x = global_mean_pool(x, batch)
        return self.mlp(x)


# Load model
gcn = GCNModel().to(device)
gcn.load_state_dict(torch.load(GCN_MODEL, weights_only=True))
gcn.eval()
print('✅ GCN model loaded')

# Load graphs and metadata
all_graphs  = torch.load(GRAPHS_FILE, weights_only=False)
test_graphs = [g for g in all_graphs if g.split == 'test']
wrong_test  = [g for g in test_graphs if g.y.item() == 0]
meta        = pd.read_csv(META_FILE)

print(f'\nTest graphs total:  {len(test_graphs)}')
print(f'Wrong test graphs:  {len(wrong_test)}  <- GNNExplainer runs on these')
print(f'Correct test graphs:{len(test_graphs) - len(wrong_test)}')

# Check what model predicts
preds = []
with torch.no_grad():
    for g in test_graphs:
        batch = torch.zeros(g.x.shape[0], dtype=torch.long)
        out   = gcn(g.x, g.edge_index, batch)
        preds.append(out.argmax(dim=1).item())

n_caught = sum(1 for g, p in zip(test_graphs, preds) if g.y.item()==0 and p==0)
print(f'\nWrong traces model caught: {n_caught} out of {len(wrong_test)}')
print('\nGNNExplainer runs on ALL wrong traces regardless of model prediction.')
print('This shows explanation output for genuine LLM reasoning failures.')


In [ ]:
explainer = Explainer(
    model=gcn,
    algorithm=GNNExplainer(epochs=200),
    explanation_type='phenomenon',   # 'phenomenon' requires target — correct for our use
    node_mask_type='attributes',
    edge_mask_type='object',
    model_config=dict(
        mode='multiclass_classification',
        task_level='graph',
        return_type='raw'
    )
)

print('✅ GNNExplainer configured')
print()
print('Settings:')
print('  Algorithm:        GNNExplainer (200 epochs per graph)')
print('  Explanation type: phenomenon — explains specific prediction with target')
print('  Node mask:        attributes — importance per node feature')
print('  Edge mask:        object — importance per edge')
print('  Task level:       graph — graph-level classification')


In [ ]:
import time

explanations = []
start = time.time()

print(f"Running GNNExplainer on {len(wrong_test)} wrong traces...")
print()

for idx, graph in enumerate(wrong_test):
    try:
        # Run GNNExplainer
        explanation = explainer(
            x          = graph.x,
            edge_index = graph.edge_index,
            batch      = torch.zeros(graph.x.shape[0], dtype=torch.long),
            target     = graph.y
        )

        # Get node importance scores
        # node_mask shape: (n_nodes, n_features) — average across features
        node_importance = explanation.node_mask.abs().mean(dim=1).numpy()
        node_importance = node_importance / (node_importance.sum() + 1e-9)  # normalize

        # Find most important step
        most_important_idx  = int(node_importance.argmax())
        most_important_score = float(node_importance.max())

        # Get step texts for this trace from metadata
        trace_steps = meta[meta['trace_id'] == graph.trace_id].sort_values('step_num')
        step_texts  = trace_steps['step_text'].tolist()
        question    = trace_steps['question'].iloc[0] if len(trace_steps) > 0 else ''

        # Build per-step results
        step_results = []
        for i, (score, text) in enumerate(zip(node_importance, step_texts)):
            step_results.append({
                'step_num':   i + 1,
                'step_text':  text,
                'importance': float(score),
                'is_failing': i == most_important_idx
            })

        explanations.append({
            'trace_id':            graph.trace_id,
            'question':            question[:150],
            'n_steps':             graph.x.shape[0],
            'failing_step_num':    most_important_idx + 1,
            'failing_step_text':   step_texts[most_important_idx] if most_important_idx < len(step_texts) else '',
            'failing_step_score':  most_important_score,
            'all_step_scores':     [float(s) for s in node_importance],
            'steps':               step_results
        })

        print(f"  [{idx+1}/{len(wrong_test)}] trace_id={graph.trace_id} | "
              f"failing step: Step {most_important_idx+1} | "
              f"score: {most_important_score:.3f}")

    except Exception as e:
        print(f"  [{idx+1}/{len(wrong_test)}] trace_id={graph.trace_id} — skipped: {e}")

elapsed = time.time() - start
print()
print(f"✅ Done in {elapsed/60:.1f} minutes")
print(f"   Explained: {len(explanations)} wrong traces")

# Save to JSON
class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, (np.integer, np.int64, np.int32)):
            return int(obj)
        if isinstance(obj, (np.floating, np.float64)):
            return float(obj)
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        return super().default(obj)

with open(EXPL_FILE, 'w') as f:
    json.dump(explanations, f, indent=2, cls=NumpyEncoder)
print(f"   Saved: {EXPL_FILE}")

In [ ]:
with open(EXPL_FILE) as f:
    explanations = json.load(f)

print(f"Showing detailed explanation for 5 wrong traces")
print()

for expl in explanations[:5]:
    print("=" * 70)
    print(f"Trace ID: {expl['trace_id']}  |  Steps: {expl['n_steps']}")
    print(f"Question: {expl['question'][:100]}...")
    print()
    print("Step-by-step importance scores:")
    for step in expl['steps']:
        marker = " ← FAILING STEP" if step['is_failing'] else ""
        bar    = "█" * int(step['importance'] * 30)
        print(f"  Step {step['step_num']} [{step['importance']:.3f}] {bar}{marker}")
        print(f"          {step['step_text'][:80]}")
    print()
    print(f"Most responsible step: Step {expl['failing_step_num']}")
    print(f"Failing step text: {expl['failing_step_text'][:100]}")
    print()

In [ ]:
from collections import Counter

failing_positions = [e['failing_step_num'] for e in explanations]
position_counts   = Counter(failing_positions)

n_total = len(explanations)
max_steps = max(e['n_steps'] for e in explanations)
step_availability = {k: sum(1 for e in explanations if e['n_steps'] >= k) for k in range(1, max_steps + 1)}

pos_scores = {k: [] for k in range(1, max_steps + 1)}
for expl in explanations:
    for step in expl['steps']:
        pos_scores[step['step_num']].append(step['importance'])

normalised_rate = {}
for k in range(1, max_steps + 1):
    avail = step_availability.get(k, 0)
    fails = position_counts.get(k, 0)
    normalised_rate[k] = (fails / avail) if avail > 0 else 0.0

print('=' * 70)
print(f'  WHICH STEP DID THE MODEL MOST RELY ON?  (n = {n_total} wrong traces)')
print('=' * 70)
print()
print(f"{'Step':<6} {'Avail.':>7} {'Failed':>7} {'Raw %':>7} {'Norm %':>8} {'Avg imp.':>10}")
print('-' * 60)
for k in sorted(pos_scores.keys()):
    avail   = step_availability.get(k, 0)
    fails   = position_counts.get(k, 0)
    raw_pct = fails / max(n_total, 1) * 100
    norm_pct = normalised_rate[k] * 100
    avg_imp = float(np.mean(pos_scores[k])) if pos_scores[k] else 0.0
    print(f'{k:<6} {avail:>7d} {fails:>7d} {raw_pct:>6.1f}% {norm_pct:>7.1f}% {avg_imp:>10.3f}')
print()
print('Raw %     = fails / total wrong traces  (biased toward early steps because they always exist)')
print('Norm %    = fails / traces that contain this step  (base-rate-corrected)')
print('Avg imp.  = mean GNNExplainer importance for this position when present')
print()
most_common_raw  = position_counts.most_common(1)[0]
most_common_norm = max(normalised_rate.items(), key=lambda kv: kv[1])
avg_failing_pos  = float(np.mean(failing_positions))
print(f'Most common (raw):        Step {most_common_raw[0]} ({most_common_raw[1]}/{n_total}, {most_common_raw[1]/n_total*100:.0f}%)')
print(f'Most common (normalised): Step {most_common_norm[0]} ({most_common_norm[1]*100:.1f}% of traces with that step)')
print(f'Mean failing step position: {avg_failing_pos:.1f}')
print(f'Mean trace length:          {np.mean([e["n_steps"] for e in explanations]):.1f}')
print()
print(f'Reminder: with n = {n_total}, a single trace shifts a per-step rate by {100/n_total:.1f}pp. Treat results as suggestive, not conclusive.')


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('GNNExplainer — Reasoning Failure Analysis', fontsize=13, fontweight='bold')

positions = sorted(pos_scores.keys())

ax1 = axes[0]
counts = [position_counts.get(p, 0) for p in positions]
colors1 = ['#c0392b' if c == max(counts) and c > 0 else '#2e86c1' for c in counts]
bars1 = ax1.bar([f'S{p}' for p in positions], counts, color=colors1, edgecolor='white')
for bar, count in zip(bars1, counts):
    if count > 0:
        ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05, str(count),
                 ha='center', va='bottom', fontsize=9, fontweight='bold')
ax1.set_title(f'Raw failure counts (n={n_total} wrong traces)')
ax1.set_xlabel('Step position'); ax1.set_ylabel('# traces where this step was identified')
ax1.spines['top'].set_visible(False); ax1.spines['right'].set_visible(False); ax1.grid(axis='y', alpha=0.3)

ax2 = axes[1]
norm_pcts = [normalised_rate.get(p, 0.0) * 100 for p in positions]
colors2 = ['#c0392b' if c == max(norm_pcts) and c > 0 else '#2e86c1' for c in norm_pcts]
bars2 = ax2.bar([f'S{p}' for p in positions], norm_pcts, color=colors2, edgecolor='white')
for bar, pct, p in zip(bars2, norm_pcts, positions):
    if step_availability.get(p, 0) > 0:
        ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                 f'{pct:.0f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax2.set_title('Normalised failure rate per position\n(fails / traces containing that step)')
ax2.set_xlabel('Step position'); ax2.set_ylabel('Failure rate (%)')
ax2.spines['top'].set_visible(False); ax2.spines['right'].set_visible(False); ax2.grid(axis='y', alpha=0.3)

ax3 = axes[2]
expl = explanations[0]
step_nums = [s['step_num'] for s in expl['steps']]
step_scores = [s['importance'] for s in expl['steps']]
step_colors = ['#c0392b' if s['is_failing'] else '#2e86c1' for s in expl['steps']]
bars3 = ax3.bar([f'S{n}' for n in step_nums], step_scores, color=step_colors, edgecolor='white')
for bar, score in zip(bars3, step_scores):
    ax3.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
             f'{score:.2f}', ha='center', va='bottom', fontsize=8)
ax3.set_title(f'Example trace (id={expl["trace_id"]})\nRed = identified step (S{expl["failing_step_num"]})')
ax3.set_xlabel('Step'); ax3.set_ylabel('Normalised importance')
ax3.spines['top'].set_visible(False); ax3.spines['right'].set_visible(False); ax3.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(CHART_FILE, bbox_inches='tight', dpi=150)
plt.show()
print(f'Chart saved: {CHART_FILE}')


In [ ]:
with open(EXPL_FILE) as f:
    explanations = json.load(f)

print("=" * 55)
print("  STAGE 7 QUALITY CHECKS")
print("=" * 55)

all_pass = True

# Check 1: Explanations generated
print(f"\n[1] Explanations generated: {len(explanations)}")
if len(explanations) > 0:
    print("    ✅ PASS")
else:
    print("    ❌ FAIL — no explanations")
    all_pass = False

# Check 2: All have failing step
has_failing = sum(1 for e in explanations if e['failing_step_num'] > 0)
print(f"\n[2] Explanations with identified failing step: {has_failing}/{len(explanations)}")
if has_failing == len(explanations):
    print("    ✅ PASS")
else:
    print("    ⚠️  Some missing")
    all_pass = False

# Check 3: Importance scores sum to ~1
bad_sums = sum(1 for e in explanations
               if abs(sum(e['all_step_scores']) - 1.0) > 0.1)
print(f"\n[3] Explanations with non-normalised scores: {bad_sums}")
if bad_sums == 0:
    print("    ✅ PASS — all scores normalised")
else:
    print("    ⚠️  Some scores not normalised")

# Check 4: Output files
print(f"\n[4] Output files:")
for path, name in [(EXPL_FILE,'explanations.json'), (CHART_FILE,'explanation_chart.png')]:
    if os.path.exists(path):
        size = os.path.getsize(path)/1024
        print(f"    ✅ {name} ({size:.0f} KB)")
    else:
        print(f"    ❌ {name} missing")
        all_pass = False

# Check 5: Summary stats
avg_score  = np.mean([e['failing_step_score'] for e in explanations])
avg_pos    = np.mean([e['failing_step_num'] for e in explanations])
print(f"\n[5] Summary statistics:")
print(f"    Avg failing step importance score: {avg_score:.3f}")
print(f"    Avg failing step position:         {avg_pos:.1f}")
if avg_score > 0.1:
    print("    ✅ PASS — explainer producing meaningful scores")
else:
    print("    ⚠️  Scores very low — check explainer")
    all_pass = False

print()
print("=" * 55)
if all_pass:
    print("  ✅ ALL CHECKS PASSED — Stage 7 complete!")
    print("  ✅ Full pipeline complete — all 7 stages done!")
else:
    print("  ⚠️  SOME CHECKS FAILED — see above")
print("=" * 55)

In [ ]:
import json

with open(os.path.join(DATA_DIR, 'results.json')) as f:
    results = json.load(f)
with open(EXPL_FILE) as f:
    explanations = json.load(f)

all_g = torch.load(GRAPHS_FILE, weights_only=False)
n_total = len(all_g)
n_correct_g = sum(1 for g in all_g if g.y.item()==1)
n_wrong_g   = sum(1 for g in all_g if g.y.item()==0)
n_train = sum(1 for g in all_g if g.split=='train')
n_val   = sum(1 for g in all_g if g.split=='val')
n_test  = sum(1 for g in all_g if g.split=='test')
meta_all = pd.read_csv(META_FILE)
n_steps  = len(meta_all)

print('=' * 78)
print('  REASONING-AS-GRAPHS — PROJECT SUMMARY')
print('  Machine Learning with Graphs | Spring 2026')
print('=' * 78)

print('\nDATASET:')
print(f'  Source:        GSM8K')
print(f'  LLM:           llama-3.1-8b-instant (Groq, temperature=0.6)')
print(f'  Total traces:  {n_total} ({n_correct_g} correct, {n_wrong_g} wrong; {n_correct_g/n_total*100:.0f}% / {n_wrong_g/n_total*100:.0f}%)')
print(f'  Total steps:   {n_steps:,}')
print(f'  Splits:        {n_train} train / {n_val} val / {n_test} test')

print('\nRESULTS (default 0.5 threshold; mean over 3 seeds):')
print(f"{'Model':<28} {'Acc':>8} {'F1':>8} {'AUC-ROC':>10} {'PR-AUC':>10}")
def line(name, m):
    return f"{name:<28} {m['accuracy']:>8.3f} {m['f1']:>8.3f} {m['auc_roc']:>10.3f} {m.get('pr_auc', float('nan')):>10.3f}"
print(line('Text Baseline',          results['text_baseline']))
print(line('Graph Feature Baseline', results['graph_feature_baseline']))
print(line('GCN',                    results['gcn']))
print(line('GAT',                    results['gat']))

print('\nINTERPRETATION:')
gf_auc  = results['graph_feature_baseline']['auc_roc']
gcn_auc = results['gcn']['auc_roc']
gat_auc = results['gat']['auc_roc']
print(f'  Best AUC-ROC:  Graph Feature Baseline ({gf_auc:.3f}) — simple structural features rank as well or better than the GNNs.')
print(f'  GCN AUC-ROC:   {gcn_auc:.3f}   |  GAT AUC-ROC: {gat_auc:.3f}')
print('  Treat between-method differences as small given 3 seeds and ~30 test graphs.')

print('\nGNNEXPLAINER:')
avg_pos   = float(np.mean([e['failing_step_num'] for e in explanations]))
avg_score = float(np.mean([e['failing_step_score'] for e in explanations]))
print(f'  Wrong traces explained:        {len(explanations)}')
print(f'  Mean failing step position:    {avg_pos:.1f}  (chains are ~6.9 steps long)')
print(f'  Mean failing step importance:  {avg_score:.3f}')
print('  Note: failing-step claims are model-interpretation, not human-labeled ground truth.')

print('\n' + '=' * 78)
print('  END OF SUMMARY')
print('=' * 78)
